# Recording a dataset

`g1lab.Recorder` writes the robot's sensors to a folder, and `g1lab.Recording`
reads them back, all on one clock, so you can ask what the joints were doing at
the moment a picture was taken.

**Recording needs no session and does not move the robot.** Part 2 moves the arm
so there is something to see, and that part does need a turn. In the lab, the
recording lands on the computer running your program, which must stay on the Robot
Network the whole time.

Run it on the mock first.

In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt
import g1lab

# "mock" runs the robot on this laptop (needs `pip install mujoco`).
# In the lab, the robot computer: HOST = "192.168.123.222"
HOST = os.environ.get("G1LAB_HOST", "mock")
NAME = os.environ.get("G1LAB_NAME", "student")     # your name: it shows on the lab board

r = g1lab.mock(name=NAME, camera="render") if HOST == "mock" else g1lab.connect(HOST, name=NAME)
info = r.info()
print("connected to", HOST, "| session:", info.phase, "| has the robot:", info.owner)

## 1. Record for a few seconds

`run(seconds)` records and gives back the `Recording`. Printing it shows what it
holds, per sensor. Each new recording goes in a new folder, named
`recording_<date>_<time>` unless you give `Recorder` one.

In [ ]:
run = g1lab.Recorder(r).run(5)
print(run)

On the robot, count on about 1.5 MB a second, so a minute is about 90 MB. Depth is
left out by default and adds about 0.9 MB a second: to choose what to record, pass
`topics`, such as `g1lab.Recorder(r, topics=["joints", "camera", "depth"])`.

## 2. Record while your program does something

Put a `with` around the code. Here the right arm swings forward and back during
the recording.

In [ ]:
home = info.home["right_arm"].copy()
forward = home.copy()
forward[0] -= 0.8                            # shoulder_pitch: forward

with g1lab.Recorder(r) as rec:
    with r.control() as c:
        c.arms.move_to(right=forward, duration=2.0)
        c.arms.move_to(right=home, duration=2.0)
print("wrote", rec.out)

## 3. Read it back

Any folder opens with `g1lab.Recording(folder)`, later or on another computer.
`read(topic)` goes through a sensor's readings as `(t, value)`, and each value is
what the live read gives: a joint reading for `joints`, the picture for `camera`.

In [ ]:
run = g1lab.Recording(rec.out)
print("sensors:", run.topics)
print(f"{run.duration:.1f} s, {len(run.stamps('joints'))} joint readings, {len(run.stamps('camera'))} pictures")

times, pitch = [], []
for t, js in run.read("joints"):
    times.append(t - run.t0)                 # every t is the robot computer's clock
    pitch.append(js.arm("right")[0])

plt.figure(figsize=(8, 3)); plt.plot(times, np.degrees(pitch))
plt.xlabel("time in the recording (s)"); plt.ylabel("right shoulder pitch (deg)"); plt.grid(alpha=.3)
plt.show()

## 4. Line the sensors up

Sensors run at different rates. `at(topic, t, max_age)` gives the newest reading at
or before `t` as `(t, value)`, or `None` if none is that recent, so this labels each
picture with the arm's angle at that moment. A camera picture is stamped when the
robot took it, about 50 ms before it arrived.

In [ ]:
pictures = list(run.read("camera"))
some = pictures[::len(pictures) // 4][:4]

fig, ax = plt.subplots(1, len(some), figsize=(12, 2.5))
for a, (t, img) in zip(ax, some):
    found = run.at("joints", t, max_age=0.1)
    a.imshow(img); a.axis("off")
    a.set_title("no joints" if found is None else f"{t - run.t0:.1f} s, {np.degrees(found[1].arm('right')[0]):.0f} deg")
plt.show()

`play(topics)` goes through several sensors in time order as `(t, topic, value)`,
which is how you replay a recording through code written for the live robot.

In [ ]:
for k, (t, topic, value) in enumerate(run.play(["joints", "camera"])):
    if k == 8:
        break
    print(f"{t - run.t0:6.3f} s  {topic}")
run.close()

From a terminal, `python -m g1lab.record --host 192.168.123.222 --seconds 60 --out
hallway` records and `python -m g1lab.record show hallway` prints what a folder holds.

**Try this.** Use `at()` to get the shoulder angle at each picture's time, plot those
points over the curve from part 3, and see how few readings the camera rate leaves you.

In [ ]:
r.close()